# Cleaning

In [1]:
%pip install Sastrawi
%pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 6.7 MB/s eta 0:00:00


In [2]:
import pandas as pd
import re
import requests
import io
import emoji
from collections import Counter
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from tqdm.auto import tqdm

In [3]:
# Ini yang baru di clean manual masih ada emoticon
df_data = pd.read_csv("https://drive.google.com/uc?id=1ofWLi8pc6vKgMpog1kqnkBsoQGVePDf3")

In [4]:
df_data.head(5)

,id,datetime,location,text,rating,accessibility,facility,activity,english_translation
0,68b8ed181b794a2cb4c11d4a667b9beb,2019-02-15 13:05:29,Taman Lembah DEWATA,"bagus , bersih , masih sepi jd enak 😁 …",5,neutral,neutral,positive,"good, clean, still quiet so it's nice 😁 …"
1,3b251988b82c428fb18da45dfe430264,2019-02-15 14:20:51,Situ Ciburuy,"Di sini dingin, yg mau ke sini bawa jaket ya",5,neutral,neutral,positive,"It's cold here, if anyone wants to come here, ..."
2,89cb60c13aeb47f49334f0bed5b001c3,2021-02-23 18:34:30,Orchid Forest Cikole,"enak bgt ke sini pas hari jumat, sepiiii 😁 …",5,neutral,neutral,positive,"It's really nice to come here on Friday, so qu..."
3,2efdc6a5fd6f4593acf383395e6ba813,2019-02-15 13:48:45,Curug Maribaya,sayang sekali ga boleh nyebur ke curug nya 😆 …,3,neutral,neutral,negative,Too bad we can't swim in the waterfall 😆
4,5cefb20d7c9d46a88e1d701945753711,2021-02-23 18:13:12,Ciwangun Indah Camp (CIC),Air terjun dan kebon tehnya bagus dan asri 👍 …,3,neutral,neutral,neutral,The waterfall and the tea plantation are beaut...


In [ ]:
url_alay = "https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv"
s_alay = requests.get(url_alay).content
df_alay = pd.read_csv(io.StringIO(s_alay.decode('utf-8')))

url_kbbi = "https://raw.githubusercontent.com/aryakdaniswara/kbbi-dataset-kbbi-v/main/csv/kbbi_v.csv"
s_kbbi = requests.get(url_kbbi).content
df_kbbi = pd.read_csv(io.StringIO(s_kbbi.decode('utf-8')))

# 4. Bangun Vocabulary
vocab_alay = set(df_alay['slang'].str.lower()) | set(df_alay['formal'].str.lower())
vocab_kbbi = set(df_kbbi.iloc[:, 0].str.lower())
safe_vocab = vocab_alay | vocab_kbbi

⏳ Mendownload kamus referensi...
⚙️ Membangun database kata...


In [ ]:
custom_fix = {
    'kesana': 'ke sana',
    'disana': 'di sana',
    'dimana': 'di mana',
    'karna': 'karena',
    'htm': 'harga tiket masuk',
    'outbond': "outbound",
    'rekomen': 'recommended',
    'rekomended': "recommended",
    'tmpat': 'tempat',
    'kemping': 'camping',
    'ujan': "hujan",
    "tracking": "trekking",
    "treking": "trekking",
    "track": "trekking",
    'selfi': 'selfie',
    'ngecamp': 'camping',
    "hp": "handphone",
    "rbu": "ribu",
    'recomended': 'recommended',
    'prewed': 'prewedding',
    'nginep': 'menginap',
    'ngopi': 'minum kopi',
    'nyantai': 'bersantai',
    'tuker': 'tukar',
    'hp': 'handphone',
    'byr': "bayar",
}

def normalize_text(text):
    if not isinstance(text, str): return ""

    # A. Hapus Emoji
    text = emoji.replace_emoji(text, replace='')

    # B. Lowercase
    text = text.lower()

    # C. Hapus URL dan Mention
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'@\w+', '', text)

    # D. Normalisasi Huruf Berulang
    text = re.sub(r'(.)\1{2,}', r'\1', text)

    # E. Normalisasi Akhiran 'ny' -> 'nya'
    text = re.sub(r'ny\b', 'nya', text)

    # F. Ganti Kata Menggunakan Kamus (Custom Fix)
    text = re.sub(r'[^a-z0-9\s"²-]', ' ', text)

    words = text.split()
    new_words = []
    for w in words:
        if w in custom_fix:
            new_words.append(custom_fix[w])
        else:
            new_words.append(w)
    text = " ".join(new_words)

    # G. Gabung kata + nya
    text = re.sub(r'(\w+)\s+nya\b', r'\1nya', text)

    # H. Gabung kata + an
    text = re.sub(r'([a-z0-9]+)[^\w\s"2²]+\s*an\b', r'\1an', text) # Tanda baca
    text = re.sub(r'([a-z0-9]+)\s+an\b', r'\1an', text)           # Spasi biasa

    # I. Gabung awalan 'nge'
    text = re.sub(r'\bnge\s+(\w+)', r'nge\1', text)

    # J. Reduplikasi (jalan2 -> jalan-jalan)
    text = re.sub(r'([a-z]+)["2²]\s*an\b', r'\1-\1an', text)
    text = re.sub(r'([a-z]+)["2²]', r'\1-\1', text)

    # K. Final cleaning spasi
    text = re.sub(r'\s+', ' ', text).strip()

    return text

tqdm.pandas()
df_data['text_cleaned'] = df_data['text'].progress_apply(normalize_text)



100%|██████████| 6000/6000 [00:01<00:00, 3672.28it/s]


In [ ]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()
stem_cache = {}

In [ ]:
# 1. Konfigurasi Filter
english_whitelist = {
    'adventure', 'area', 'back', 'bad', 'bag', 'ball', 'beautiful', 'best', 'budget', 'camp',  'camping',
    'charge', 'city', 'clean', 'court', 'cozy', 'cz', 'family', 'feeding', 'finishing', 'food',
    'for', 'forest', 'fresh', 'flying', 'fox', 'friendly', 'gate', 'geopark', 'glamping', 'golden', 'good', 'google',
    'great', 'hammock', 'happy', 'healing', 'heaven', 'hiking', 'inc', 'indoor', 'instagramable', 'is', 'it',
    'music', 'nature', 'next', 'nice', 'of', 'one', 'only', 'out', 'outdoor', 'paint',
    'park', 'place', 'playground', 'rabbit', 'recommended', 'refresh', 'refund', 'resort', 'secret', 'service',
    'shuttle', 'spot', 'station', 'sunrise', 'stone', 'sunset', 'team', 'the', 'things', 'this', 'ticket',
    'time', 'trekking', 'view', 'villa', 'visit', 'weekend', 'weekday', 'welcome', 'worth', 'zoo',
    'refreshing', 'rainbow', 'outbound', 'floating', 'garden', 'selfie', 'market', 'cafe', 'gathering', 'free', 'slide',
    'orchid', 'overall', 'prewedding', 'live', 'and', 'ground', 'hills', 'hunting',
    'amazing', 'guys', 'drink', 'instagram', 'liberty', 'strawberry', 'guide', 'all',
    'haha', 'ok', 'handphone', "recommend"
}

indonesian_whitelist = {
    'tangkuban', 'lembang', 'ciwidey', 'bandung', 'cikole',
    'dll', 'maribaya', 'sanghyang', 'tilu', 'musholla', 'parahu', 'sarae',
    'padalarang', 'punclut', 'cic', 'malela', 'dago', 'santorini',
    'ngadem', 'nanjak', 'wkwk', 'stand', 'warni'
}

# Update safe_vocab dengan SEMUA whitelist
current_safe_vocab = safe_vocab.copy()
current_safe_vocab.update(english_whitelist)
current_safe_vocab.update(indonesian_whitelist)

unknown_words = Counter()

def smart_scan(text):
    if not isinstance(text, str): return

    # Bersihkan sisa tanda baca
    text = re.sub(r'[^a-z\s]', ' ', text)
    words = text.split()

    for w in words:

        if w in current_safe_vocab: continue

        # Cek yang Suffix "nya"
        if w.endswith('nya'):
            base_nya = w[:-3] # Buang 3 huruf terakhir ('nya')
            if base_nya in current_safe_vocab:
                continue

        # Stemming dengan Sastrawi
        if w not in stem_cache:
            stem_cache[w] = stemmer.stem(w)

        root_word = stem_cache[w]

        # Cek Kamus lagi untuk kata dasar hasil stemming
        if root_word in current_safe_vocab: continue

        # Jika lolos semua, catat sebagai unknown
        unknown_words[w] += 1

tqdm.pandas()
df_data['text_cleaned'].progress_apply(smart_scan)

print(f"\nTotal kata unik yang TIDAK DIKENAL: {len(unknown_words)}")
print("--- 50 Kata Asing Terbanyak Muncul (Top Frequency) ---")
# Saya set reverse=True agar yang muncul di atas adalah yang paling sering (masalah utama)
sorted_unknown = sorted(unknown_words.items(), key=lambda x: x[1], reverse=True)

for word, count in sorted_unknown[:50]:
    print(f"{word} : {count}")

100%|██████████| 6000/6000 [00:00<00:00, 51070.43it/s]


Total kata unik yang TIDAK DIKENAL: 3105
--- 50 Kata Asing Terbanyak Muncul (Top Frequency) ---
an : 17
lho : 15
explore : 15
heuleut : 15
eiffel : 15
ngecamp : 14
nan : 14
ciburuy : 14
siapin : 14
pawon : 14
kenit : 14
weekdays : 14
saguling : 12
terimakasih : 12
muter : 12
bantuin : 12
masing : 12
btw : 12
domas : 12
jayagiri : 12
selfi : 12
pikuk : 11
hahaha : 11
maps : 11
covid : 10
dituker : 10
tour : 10
segitu : 10
enjoy : 10
laper : 9
wedding : 9
rajamandala : 9
rbu : 9
cililin : 9
dslr : 9
holiday : 9
game : 9
room : 9
stroller : 9
bond : 8
bro : 8
extra : 8
leuwi : 8
pemandanganya : 8
icon : 8
burangrang : 8
voucher : 8
hits : 8
ig : 8
masya : 8


In [ ]:
df_data['text'] = df_data['text_cleaned']
df_data.drop(columns=['text_cleaned'], inplace=True)
output_filename = 'dataset_cleaned_final.csv'
df_data.to_csv(output_filename, index=False)